In [ ]:
import pandas as pd

from ada_fsaudit_bridge import (
    att_sample,
    configure_environment,
    cvs_sample,
    load_dataset,
    lower_bound,
    mus_sample,
    set_notebook_context,
    upper_bound,
)
from scipy.stats import hypergeom

ADA_WORKSHOP_ID = "hypothesis-testing"
ADA_CHAPTER = "4"
ADA_NOTEBOOK_SOURCE = "notebooks/support/hypothesis-testing/support.Rmd"

def ada_set_context(exercise_ref: str) -> None:
    set_notebook_context(
        chapter=ADA_CHAPTER,
        exercise=exercise_ref,
        notebook=f'{ADA_WORKSHOP_ID}:{ADA_NOTEBOOK_SOURCE}',
    )

configure_environment()


## Exercise 4.1. Effect of the critical region on the sample size


The results on page 119 are obtained with the `FSaudit` package, first by creating an attribute object, filling it with the parameters of the test, and then applying the `size()` function.
The relevant parameters are the significance level `alpha`, the number of deviations in the population $M$ `popdev`, the population size $N$ `popn`, and the critical region `c`.


In [ ]:
ada_set_context("4.1")
mySample = att_sample(alpha=0.1, popdev=60, popn=1200, c=0).size()
mySample.n


The sample size increases as we increase the critical region to `c = 2`.


In [ ]:
ada_set_context("4.1")
mySample.size(c=2).n


This result is obtained using the default Hypergeometric distribution. To obtain the sample size with the binomial approximation, as in Table 5.3, we specify the distribution when setting up the attribute object.


In [ ]:
ada_set_context("4.1")
mySample2 = att_sample(alpha=0.1, tdr=0.05, c=2, dist="binom").size()
mySample2.n


## Exercise 4.2. Significance levels


On page 120 we calculated significance levels for the occurrence of finding one or two errors. These probabilities are calculated in `R` with the following code:


In [ ]:
hypergeom.cdf(1, 1200, 60, 45)
hypergeom.cdf(2, 1200, 60, 102)


Refer to Equation 2.1.
Remember that `R` uses `q` for the number of errors found $k$, `m` for the number of deviations in the population $M$, `n` for the number of correct items $N - M$, and `k` for the sample size $n$.


## Exercise 4.3. Type II error


On page 121 we also calculated the Type II error, for a scenario with no errors allowed in the sample and 24 errors in the population, or a population error rate of $24 / 200 = 0.02$.


In [ ]:
hypergeom.sf(0, 1200, 24, 45)


## Exercise 4.4. One-sided upper bounds


The one-sided upper bounds $p_U$ in Table 5.1 are obtained as follows:


In [ ]:
from IPython.display import display
display(upper_bound(k=0, popn=1200, n=102, alpha=0.10) / 1200)
display(upper_bound(k=3, popn=1200, n=102, alpha=0.10) / 1200)


## Exercise 4.5. Case: European innovation subsidies


The sample size in the *Case: European innovation subsidies* depends on the critical region chosen. When the null hypothesis $H_0 : M \geq 120,000$ is rejected when the sample yields no errors, the critical region is $\{k | k = 0\}$. We first create an `mus_obj` object, and load it with the parameters.


In [ ]:
ada_set_context("4.5")
subsidies = mus_sample(cl=0.95, popBv=12000000, pm=120000).size(ee=0)
subsidies.n


This result is exactly equal to that of the fixed-attribute sample:


In [ ]:
ada_set_context("4.5")
myAttSample = att_sample(alpha=0.05, popn=12000000, popdev=120000).size(c=0)
myAttSample.n


To build a margin for one error, we may increase the critical region to $\{k | k \leq 1\}$, resulting in a sample size of $n =$ 473.


In [ ]:
ada_set_context("4.5")
myAttSample.size(c=1).n


In MUS, we increase the critical region by anticipating on the expected error (in monetary terms) in the population. Thus, in applications where selected items are either completely correct or completely incorrect, it is easier to calculate the minimum required sample size using the `att_obj` than using the `mus_obj`.


## Exercise 4.6. Case: Accounts receivable circularization


We start by setting up the MUS object `ar` (for Accounts Receivable) in `R` with the `FSaudit` package and first verify that the number of sampling units `popn` and the total book value `popBv` of the sampling frame match those in the population. Notice that these statistics are calculated as soon as the object is loaded with the detail amounts.


In [ ]:
ada_set_context("4.6")
accounts_receivable = load_dataset("accounts_receivable")
ar = mus_sample(
  bv=accounts_receivable["amount"],
  id=accounts_receivable["invoice"],
)
ar.popn, ar.popBv


Sample size calculation is invoked with the relevant values from the *Case: Accounts receivable circularization*.


In [ ]:
ada_set_context("4.6")
ar.size(cl=0.95, pm=450000, ee=100000, evalMeth="Stringer").n


Compare this with the sample size calculated using fixed-attribute sampling.


In [ ]:
from IPython.display import display
ada_set_context("4.6")
ar2 = att_sample(alpha=0.05, popn=13500000, popdev=450000).size(c=1)
display(ar2.n)
display(ar2.size(c=2).n)


We can therefore infer that with an expected error of `ee = 100000`, we can tolerate between one and two 100% errors.


## Exercise 4.7. Multiple hits with random selection


If we draw a sample of size $n = 1,100$ from the `accounts_receivable` population, the largest sampling units have an almost 100% inclusion probability and with a selection method such as `random` are likely to be hit more than once.
For example, the data file `accounts_receivable`, that comes with the `FSaudit` package, has the following six largest amounts:


In [ ]:
ada_set_context("4.7")
accounts_receivable.sort_values("amount", ascending=False).iloc[:6, :3]


We set up a new `mus_obj` and use a materiality of `pm = 36730` to arrive at a sample size of 1100.


In [ ]:
ada_set_context("4.7")
multiple = mus_sample(
  bv=accounts_receivable["amount"],
  id=accounts_receivable["invoice"],
  pm=36730,
).size()
multiple.n


We select the sample with the selection method `selMeth = "random"` and order it in decreasing order, displaying the six largest book values.


In [ ]:
ada_set_context("4.7")
multiple.select(selMeth="random", seed=1)
sample = multiple.sample
sample.sort_values("bv", ascending=False).head()


This example demonstrates that invoices 201719763 and 201710344 were selected twice.


## Exercise 4.8. Stringer bound


We continue the case study sample, and select the sample, now with the "randomized fixed" selection method.


In [ ]:
ada_set_context("4.8")
ar.select(selMeth="randomized.fixed", seed=345)
ar.sample.head()


Before we can evaluate the sample, we must first provide audit values. These should be provided in a list of the same order as the list of sample book values. We first copy the list of book values into a data frame.


In [ ]:
ada_set_context("4.8")
myResults = ar.sample.loc[:, ["item", "bv"]].rename(columns={"bv": "av"}).copy()
myResults


Table 5.4. lists the three invoices (items 16, 52, and 124), that are assumed to be in error.


In [ ]:
ada_set_context("4.8")
myResults.iloc[[15, 51, 123], :]


We update the audit values of the erroneous items.


In [ ]:
ada_set_context("4.8")
myResults.iloc[15, 1] = 4438.82
myResults.iloc[51, 1] = 0
myResults.iloc[123, 1] = 5531.38
myResults.iloc[[15, 51, 123], :]


The updated list of audit values is then submitted to the MUS object for evaluation.


In [ ]:
ada_set_context("4.8")
ar.evaluate(av=myResults["av"], evalMeth="stringer")


The results of the Stringer bound evaluation as presented in Tables 5.6 and 5.7 are stored in the `Precision calculation` attribute.


In [ ]:
ada_set_context("4.8")
ar.eval_results["Over"]["Precision calculation"]


The upper bounds $M_U$ in Table 5.5 are calculated with the `upper` function from the `FSaudit` package.


In [ ]:
upper_bound(k=0, popn=13500000, n=145, alpha=0.05)
upper_bound(k=1, popn=13500000, n=145, alpha=0.05)
upper_bound(k=2, popn=13500000, n=145, alpha=0.05)
upper_bound(k=3, popn=13500000, n=145, alpha=0.05)


## Exercise 4.9. Cell evaluation


The results of the cell evaluation method, as presented in Table 5.8, can also be obtained, by changing the evaluation method.


In [ ]:
from IPython.display import display
ada_set_context("4.9")
pd.set_option("display.width", 70)
display(ar.evaluate(av=myResults["av"], evalMeth="cell"))
display(ar.eval_results["Over"]["Precision calculation"])


## Exercise 4.10. PPS estimation


Finally, we present results from the `pps` evaluation method. For this purpose, we use audit values stored in the variable `av2`. The total error amount reflected in `av2` is 450,000.


In [ ]:
from IPython.display import display
ada_set_context("4.10")
myResults["av2"] = accounts_receivable.set_index("invoice").loc[myResults["item"], "av2"].to_numpy()
display(ar.evaluate(av=myResults["av2"], evalMeth="pps"))
display(ar.eval_results["Error estimate"])


A two-sided prediction interval around the PPS estimate is calculated according to Equations 5.6 and 5.7.


In [ ]:
from IPython.display import display
ada_set_context("4.10")
display(ar.eval_results["pps estimate"])
display(ar.eval_results["Lower bound"])
display(ar.eval_results["Upper bound"])
